# Tutorial 05: Streaming Recursive Least Squares & Quantile Regression

### Learning Objectives
1. Master online updating via Recursive Least Squares (RLS) with $\mathcal{O}(D^2)$ time/memory.
2. Track non-stationary concept drift using the exponential forgetting factor $\lambda$.
3. Formulate the asymmetric pinball loss for Quantile Regression:
   $$\rho_\tau(u) = u (\tau - \mathbb{I}(u < 0))$$
4. Build asymmetric risk ribbons (10th percentile floor, median, 90th percentile ceiling).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Robust import setup: traverse up until 'src' directory is found
root_dir = Path.cwd().resolve()
while not (root_dir / "src").exists() and root_dir != root_dir.parent:
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.streaming import RecursiveLeastSquares
from src.quantile import QuantileRegressorScratch

## 1. Online Learning with RLS under Concept Drift

In [ ]:
np.random.seed(42)
N = 1000
X = np.random.randn(N, 2)
y = np.zeros(N)

# Dynamic regime shift at t = 500
for t in range(N):
    if t < 500:
        y[t] = 2.0 * X[t, 0] - 1.5 * X[t, 1] + 0.5 + np.random.normal(0, 0.1)
    else:
        y[t] = -1.0 * X[t, 0] + 3.0 * X[t, 1] - 0.5 + np.random.normal(0, 0.1)

rls = RecursiveLeastSquares(lambda_=0.98)
w0_history = []

for t in range(N):
    rls.partial_fit(X[t], y[t])
    w0_history.append(float(rls.weights[0]))

print(f"RLS adapted weights after drift: w0={rls.weights[0]:.4f} (ground truth: -1.0), w1={rls.weights[1]:.4f} (ground truth: 3.0)")

## 2. Quantile Regression on Heteroscedastic Data

In [ ]:
np.random.seed(42)
N = 300
x_fan = np.random.uniform(0.5, 5.0, N)
# Error variance expands with x (heteroscedastic fan)
noise_fan = np.random.normal(0, 0.3 * x_fan, N)
y_fan = 2.0 * x_fan + noise_fan
X_fan = x_fan.reshape(-1, 1)

# Fit 10th percentile, Median (50th), and 90th percentile
q10 = QuantileRegressorScratch(quantile=0.10).fit(X_fan, y_fan)
q50 = QuantileRegressorScratch(quantile=0.50).fit(X_fan, y_fan)
q90 = QuantileRegressorScratch(quantile=0.90).fit(X_fan, y_fan)

x_grid = np.linspace(0.5, 5.0, 100).reshape(-1, 1)
pred_10 = q10.predict(x_grid)
pred_50 = q50.predict(x_grid)
pred_90 = q90.predict(x_grid)

# Plotting the asymmetric risk corridor
plt.figure(figsize=(9, 4.5))
plt.scatter(x_fan, y_fan, color="tab:blue", alpha=0.6, label="Heteroscedastic Observations")
plt.plot(x_grid, pred_50, color="black", lw=2, label="Median Fit (τ = 0.50)")
plt.fill_between(x_grid[:, 0], pred_10, pred_90, color="tab:orange", alpha=0.3, label="80% Quantile Corridor [τ=0.10, τ=0.90]")
plt.title("Quantile Regression: Asymmetric Risk Corridor on Heteroscedastic Data", fontweight="bold")
plt.xlabel("Feature x")
plt.ylabel("Target y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()